<a href="https://colab.research.google.com/github/nehnamehranmk638-dev/multilingual-rag-research/blob/nehna/notebooks/ml_05_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Malayalam Phase 1 — Step 5: Generation

Mirrors English `05_generation.ipynb`.

**What this notebook does:**
1. Loads Qwen2.5-1.5B-Instruct (same model as English)
2. Runs generation on 100 Malayalam questions using hybrid retrieval results
3. Evaluates: EM, F1, Contains-match
4. Runs gold-passage ceiling experiment
5. Saves `predictions_hybrid_ml.json`, `predictions_gold_ml.json`, `generation_experiments_ml.csv`

## 1. GitHub setup

In [2]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

print("Token loaded:", token is not None)

Token loaded: True


In [3]:
import os

os.environ["GITHUB_TOKEN"] = token

!git clone https://$GITHUB_TOKEN@github.com/nehnamehranmk638-dev/multilingual-rag-research.git

Cloning into 'multilingual-rag-research'...
remote: Enumerating objects: 217, done.
remote: Counting objects: 100% (217/217), done.
remote: Compressing objects: 100% (181/181), done.
remote: Total 217 (delta 102), reused 129 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (217/217), 1.18 MiB | 4.65 MiB/s, done.
Resolving deltas: 100% (102/102), done.


In [4]:
%cd /content/multilingual-rag-research

/content/multilingual-rag-research


In [5]:
import subprocess

username = "nehnamehrankmk638-dev"

credential = f"""protocol=https
host=github.com
username={username}
password={token}

"""

subprocess.run(
    ["git", "credential", "approve"],
    input=credential,
    text=True,
    check=True
)

print("GitHub authentication configured.")

GitHub authentication configured.


In [6]:
!git fetch origin
!git switch nehna
!git pull

branch 'nehna' set up to track 'origin/nehna'.
Switched to a new branch 'nehna'
Already up to date.


## 2. Load data

In [7]:
import json

with open("data/questions_ml.json", "r", encoding="utf-8") as f:
    questions_ml = json.load(f)

with open("data/corpus_ml.json", "r", encoding="utf-8") as f:
    corpus_ml = json.load(f)

with open("results/hybrid_top10_ml.json", "r", encoding="utf-8") as f:
    hybrid_results_ml = json.load(f)

print("Questions   :", len(questions_ml))
print("Corpus      :", len(corpus_ml))
print("Hybrid IDs  :", len(hybrid_results_ml))

Questions   : 100
Corpus      : 247
Hybrid IDs  : 100


## 3. Install dependencies & load model

In [8]:
!pip install -q transformers accelerate

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [10]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Model loaded:", model_name)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct


## 4. Malayalam prompt & generation function

> **Note:** We use a Malayalam-language instruction prompt.
> Using English instructions with Malayalam context confuses the model
> (tested in earlier notebook — it returned wrong answers).

We also truncate passages by character limit (not token limit) to avoid
cutting the answer off mid-sentence.

In [11]:
MAX_PASSAGE_CHARS = 600   # safe limit — keeps answer within context window

def truncate_passage(text, max_chars=MAX_PASSAGE_CHARS):
    """Truncate passage at sentence boundary to preserve answer."""
    if len(text) <= max_chars:
        return text
    # Try to cut at last full stop before limit
    cut = text[:max_chars].rfind(".")
    if cut > max_chars // 2:
        return text[:cut + 1]
    return text[:max_chars]

In [12]:
def build_prompt_ml_fixed(question, passages):
  # Use top-2 passages only, each trimmed to 700 chars (safe for token budget)
  context = "\n\n".join(f"[{i+1}] {p[:700]}" for i, p in enumerate(passages[:2]))

  return f"""Answer the question in Malayalam using ONLY the context below.
Provide a short, direct answer (1 to 4 words). Do not explain.

Context:
{context}

Question: {question}

Answer:"""


def generate_answer_ml_fixed(question, passages, max_new_tokens=40):
  prompt = build_prompt_ml_fixed(question, passages)
  messages = [{"role": "user", "content": prompt}]

  text = tokenizer.apply_chat_template(
      messages, tokenize=False, add_generation_prompt=True
  )

  # No hard 2048 truncation — let it use Qwen's native context window
  inputs = tokenizer(text, return_tensors="pt").to(llm.device)

  with torch.no_grad():
    output = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.1,
    )

  generated_tokens = output[0][inputs["input_ids"].shape[1] :]
  return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


# Test on the first 5 questions
for q in questions_ml[:5]:
  top_ids = hybrid_results_ml[str(q["question_id"])][:2]
  passages = [corpus_ml[pid]["text"] for pid in top_ids]
  ans = generate_answer_ml_fixed(q["question"], passages)

  print("=" * 60)
  print("Question :", q["question"])
  print("Gold     :", q["gold_answers"])
  print("Model    :", ans)

Question : ഭരതനാട്യത്തിൽ രണ്ട് കൈകളും കാണിക്കുന്ന സംയുക്ത മുദ്രകൾ  എത്ര എണ്ണമുണ്ട് ?
Gold     : ['രണ്ട്']
Model    : 24
Question : റാണി ലക്ഷ്മിഭായ് അടിയന്തിര സഹായത്തിനായി ആരെയാണ് ബന്ധപ്പെട്ടത് ?
Gold     : ['താന്തിയോതോപ്പിയെ']
Model    : അറിയപ്പെട്ട്
Question : ഡച്ചുകാർ ചെന്നൈ പിടിച്ചെടുത്തത് ഏത് വര്ഷം ?
Gold     : ['1612-ഇൽ']
Model    : 1612
Question : ചരിത്രാതീത കാലം മുതൽ ഹിമാചൽ  പ്രദേശത്ത് വസിച്ചിരുന്ന ചില ഗോത്രങ്ങൾ ഏതെല്ലാം ?
Gold     : ['കോളി, ഹാലി, ഡാഗി']
Model    : 8
Question : ഗൂർഖയിലെ പൃഥ്വി നാരായൺ ഷാ നേപ്പാളിനെ ഏകീകരിച്ചത് ഏത് വർഷം?
Gold     : ['1765ൽ']
Model    : 1996


## 5. Sanity check on 3 examples

## 6. Run generation on all 100 questions (hybrid retrieval)

In [16]:
predictions_ml = {}

for i, q in enumerate(questions_ml):
    qid = q["question_id"]

    top_ids = hybrid_results_ml[str(qid)][:5]
    passages = [corpus_ml[pid]["text"] for pid in top_ids]

    predictions_ml[qid] = generate_answer_ml_fixed(q["question"], passages)

    if i % 10 == 0:
        print(f"{i}/100 done")

print("Done. Total predictions:", len(predictions_ml))

0/100 done
10/100 done
20/100 done
30/100 done
40/100 done
50/100 done
60/100 done
70/100 done
80/100 done
90/100 done
Done. Total predictions: 100


## 7. Evaluation metrics (EM, F1, Contains-match)

In [17]:
import re
import string

def normalize_answer_ml(text):
    """Normalize Malayalam answer for comparison.
    Unlike English, we do NOT strip articles (no articles in Malayalam).
    We strip punctuation and extra whitespace.
    """
    text = text.strip()
    # Remove common trailing punctuation
    text = re.sub(r"[?!\.,;:।]", "", text)
    text = " ".join(text.split())
    return text

def exact_match_ml(prediction, gold_answers):
    pred_norm = normalize_answer_ml(prediction)
    for gold in gold_answers:
        if pred_norm == normalize_answer_ml(gold):
            return 1
    return 0

def f1_score_ml(prediction, gold):
    pred_tokens = normalize_answer_ml(prediction).split()
    gold_tokens = normalize_answer_ml(gold).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return int(pred_tokens == gold_tokens)

    common = {}
    for t in pred_tokens:
        common[t] = common.get(t, 0) + 1

    gold_counts = {}
    for t in gold_tokens:
        gold_counts[t] = gold_counts.get(t, 0) + 1

    overlap = sum(
        min(count, common.get(token, 0))
        for token, count in gold_counts.items()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

def best_f1_ml(prediction, gold_answers):
    return max(f1_score_ml(prediction, gold) for gold in gold_answers)

def contains_match_ml(prediction, gold_answers):
    pred_norm = normalize_answer_ml(prediction)
    for gold in gold_answers:
        if normalize_answer_ml(gold) in pred_norm:
            return 1
    return 0

In [18]:
em_scores_ml = []
f1_scores_ml = []
contains_scores_ml = []

for q in questions_ml:
    pred = predictions_ml[q["question_id"]]

    em_scores_ml.append(exact_match_ml(pred, q["gold_answers"]))
    f1_scores_ml.append(best_f1_ml(pred, q["gold_answers"]))
    contains_scores_ml.append(contains_match_ml(pred, q["gold_answers"]))

em_hybrid = sum(em_scores_ml) / len(em_scores_ml)
f1_hybrid = sum(f1_scores_ml) / len(f1_scores_ml)
contains_hybrid = sum(contains_scores_ml) / len(contains_scores_ml)

print("--- Hybrid Retrieval ---")
print(f"Exact Match    : {em_hybrid:.4f}")
print(f"F1             : {f1_hybrid:.4f}")
print(f"Contains-match : {contains_hybrid:.4f}")

--- Hybrid Retrieval ---
Exact Match    : 0.0200
F1             : 0.0577
Contains-match : 0.0200


## 8. Gold-passage ceiling experiment

In [20]:
gold_predictions_ml = {}

for i, q in enumerate(questions_ml):
    gold_passage = corpus_ml[q["gold_passage_id"]]["text"]
    gold_predictions_ml[q["question_id"]] = generate_answer_ml_fixed(
        q["question"], [gold_passage]
    )

    if i % 10 == 0:
        print(f"{i}/100 done")

print("Done. Total gold predictions:", len(gold_predictions_ml))

0/100 done
10/100 done
20/100 done
30/100 done
40/100 done
50/100 done
60/100 done
70/100 done
80/100 done
90/100 done
Done. Total gold predictions: 100


In [21]:
gold_em_ml = [
    exact_match_ml(gold_predictions_ml[q["question_id"]], q["gold_answers"])
    for q in questions_ml
]
gold_f1_ml = [
    best_f1_ml(gold_predictions_ml[q["question_id"]], q["gold_answers"])
    for q in questions_ml
]

em_gold = sum(gold_em_ml) / len(gold_em_ml)
f1_gold = sum(gold_f1_ml) / len(gold_f1_ml)

print("--- Gold Passage Ceiling ---")
print(f"Exact Match : {em_gold:.4f}")
print(f"F1          : {f1_gold:.4f}")

print("\n--- Retrieval Gap ---")
print(f"EM gap  : {(em_gold - em_hybrid)*100:.1f} pp")
print(f"F1 gap  : {(f1_gold - f1_hybrid)*100:.1f} pp")

--- Gold Passage Ceiling ---
Exact Match : 0.0600
F1          : 0.1003

--- Retrieval Gap ---
EM gap  : 4.0 pp
F1 gap  : 4.3 pp


In [23]:
refusals_ml = sum(
    1
    for q in questions_ml
    if "i don't know" in str(predictions_ml[q["question_id"]]).lower()
)
print(f"Refusals: {refusals_ml} out of {len(questions_ml)}")

Refusals: 0 out of 100


## 9. Save results

In [24]:
import os

os.makedirs("results", exist_ok=True)

with open("results/predictions_hybrid_ml.json", "w", encoding="utf-8") as f:
    json.dump(predictions_ml, f, ensure_ascii=False, indent=2)

with open("results/predictions_gold_ml.json", "w", encoding="utf-8") as f:
    json.dump(gold_predictions_ml, f, ensure_ascii=False, indent=2)

print("Saved: predictions_hybrid_ml.json")
print("Saved: predictions_gold_ml.json")

Saved: predictions_hybrid_ml.json
Saved: predictions_gold_ml.json


In [25]:
import csv

with open("results/generation_experiments_ml.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["run_name", "retrieval_method", "model_name", "em", "f1", "contains_match", "refusals"])
    writer.writerow([
        "hybrid_generation_ml",
        "hybrid",
        "Qwen/Qwen2.5-1.5B-Instruct",
        round(em_hybrid, 6),
        round(f1_hybrid, 6),
        round(contains_hybrid, 6),
        refusals_ml
    ])

print("Saved: generation_experiments_ml.csv")

Saved: generation_experiments_ml.csv


## 10. Push to GitHub

In [28]:
!git add results/predictions_hybrid_ml.json \
          results/predictions_gold_ml.json \
          results/generation_experiments_ml.csv

!git commit -m "ml phase1: generation results (hybrid + gold ceiling)"
!git push

[nehna bb30b8b] ml phase1: generation results (hybrid + gold ceiling)
 3 files changed, 201 insertions(+), 201 deletions(-)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 3.72 KiB | 3.72 MiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/nehnamehranmk638-dev/multilingual-rag-research.git
   feff917..bb30b8b  nehna -> nehna


In [27]:
!git config --global user.email "nehnamehranmk17@gmail.com"
!git config --global user.name "nehnamehranmk638-dev"